# 01 · Ingesta y verificación de datos

**Fase 1 del E3.** Convertimos los 9 CSV del SCANIA Component X Dataset a Parquet, cargamos cada conjunto (lecturas + especificaciones + etiquetas) y comprobamos que los datos son coherentes antes de limpiarlos y explorarlos.

Comprobaciones:
1. Los 9 ficheros están completos (filas esperadas).
2. Cada `vehicle_id` aparece en los tres ficheros de su conjunto y no hay camiones compartidos entre train, validación y test.
3. No hay lecturas duplicadas y `time_step` crece dentro de cada camión.
4. Porcentaje de nulos por columna.
5. Contadores que decrecen (posibles reinicios de la ECU).
6. Distribución de etiquetas y coste de referencia "predecir siempre 0".

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import data

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Ficheros originales

In [ ]:
raw_status = data.check_raw(count_rows=True)
raw_status

In [ ]:
assert raw_status["ok"].all(), "Falta algún CSV o no tiene las filas esperadas" 

## 2. Conversión a Parquet

Se hace por bloques para no cargar el CSV de entrenamiento (1,14 GB) entero en memoria. Las lecturas se guardan en `float64` porque algunos contadores superan 2²⁴ y en `float32` perderían precisión. Si los Parquet ya existen, no se regeneran.

In [ ]:
data.build_parquet()

pd.DataFrame([
    {"fichero": p.name, "MB": round(p.stat().st_size / 1e6, 1)}
    for p in sorted(data.PROCESSED.glob("*.parquet"))
])

In [ ]:
csv_mb = raw_status["MB"].sum()
pq_mb = sum(p.stat().st_size for p in data.PROCESSED.glob("*.parquet")) / 1e6
print(f"CSV: {csv_mb:,.0f} MB  ->  Parquet: {pq_mb:,.0f} MB  ({pq_mb / csv_mb:.0%})")

## 3. Carga de los tres conjuntos

Para las comprobaciones cargamos las lecturas en `float32` (la mitad de memoria); el análisis posterior puede usar `float64`.

In [ ]:
splits = {s: data.load_split(s, float32=True) for s in data.SPLITS}

pd.DataFrame([
    {
        "conjunto": s,
        "camiones": r["vehicle_id"].nunique(),
        "lecturas": len(r),
        "columnas lecturas": r.shape[1],
        "lecturas/camión (mediana)": r.groupby("vehicle_id").size().median(),
        "memoria lecturas (MB)": round(r.memory_usage(deep=True).sum() / 1e6),
    }
    for s, (r, sp, lb) in splits.items()
])

## 4. Coherencia de identificadores

In [ ]:
rows = []
ids = {}
for s, (r, sp, lb) in splits.items():
    a, b, c = set(r["vehicle_id"]), set(sp["vehicle_id"]), set(lb["vehicle_id"])
    ids[s] = a
    rows.append({
        "conjunto": s,
        "ids lecturas": len(a), "ids specs": len(b), "ids etiquetas": len(c),
        "mismos ids en los 3": a == b == c,
        "ids duplicados en specs": sp["vehicle_id"].duplicated().sum(),
        "ids duplicados en etiquetas": lb["vehicle_id"].duplicated().sum(),
    })
id_check = pd.DataFrame(rows)
id_check

In [ ]:
print("train ∩ validation:", len(ids["train"] & ids["validation"]))
print("train ∩ test:      ", len(ids["train"] & ids["test"]))
print("validation ∩ test: ", len(ids["validation"] & ids["test"]))

## 5. Duplicados y orden temporal

In [ ]:
rows = []
for s, (r, _, _) in splits.items():
    dup_rows = r.duplicated().sum()
    dup_key = r.duplicated(["vehicle_id", "time_step"]).sum()
    # tras ordenar, time_step debe ser estrictamente creciente dentro de cada camión
    step = r.groupby("vehicle_id")["time_step"].diff()
    rows.append({
        "conjunto": s,
        "filas duplicadas": dup_rows,
        "(vehicle_id, time_step) repetidos": dup_key,
        "saltos time_step <= 0": int((step <= 0).sum()),
        "time_step mín": r["time_step"].min(),
        "time_step máx": r["time_step"].max(),
    })
order_check = pd.DataFrame(rows)
order_check

## 6. Valores nulos

In [ ]:
null_pct = pd.DataFrame({s: r[data.SENSOR_COLUMNS].isna().mean() * 100 for s, (r, _, _) in splits.items()})
print("Columnas con algún nulo:", (null_pct.max(axis=1) > 0).sum(), "de", len(null_pct))
print("Máximo % de nulos en una columna:", f"{null_pct.values.max():.2f} %")
null_pct.sort_values("train", ascending=False).head(10)

In [ ]:
# ¿Los nulos van por lectura completa (falta todo un sensor) o sueltos?
r = splits["train"][0]
null_by_var = pd.DataFrame({
    var: r[[c for c in data.SENSOR_COLUMNS if c.split("_")[0] == var]].isna().any(axis=1)
    for var in sorted({c.split("_")[0] for c in data.SENSOR_COLUMNS})
})
print(f"Lecturas de train con algún nulo: {null_by_var.any(axis=1).mean():.2%}")
null_by_var.mean().sort_values(ascending=False).mul(100).round(3).rename("% lecturas con nulos").to_frame()

## 7. Contadores que decrecen

Los contadores son acumulados y deberían crecer siempre. Un descenso entre dos lecturas consecutivas del mismo camión apunta a un reinicio de la ECU o a un error de registro.

In [ ]:
rows = []
for s, (r, _, _) in splits.items():
    diffs = r.groupby("vehicle_id")[data.COUNTERS].diff()
    dec = diffs < 0
    for c in data.COUNTERS:
        rows.append({
            "conjunto": s, "contador": c,
            "descensos": int(dec[c].sum()),
            "% lecturas": dec[c].mean() * 100,
            "camiones afectados": r.loc[dec[c], "vehicle_id"].nunique(),
        })
decreases = pd.DataFrame(rows).pivot(index="contador", columns="conjunto", values=["descensos", "camiones afectados"])
decreases

## 8. Etiquetas

### Entrenamiento: tiempo hasta el evento

In [ ]:
tte = splits["train"][2]
print(tte["in_study_repair"].value_counts().rename({0: "sin reparación (censurado)", 1: "con reparación"}).to_frame("camiones")
      .assign(pct=lambda d: (d["camiones"] / d["camiones"].sum() * 100).round(2)))
tte.groupby("in_study_repair")["length_of_study_time_step"].describe()

### Validación y test: clase de la última lectura

In [ ]:
classes = pd.DataFrame({s: splits[s][2]["class_label"].value_counts().sort_index() for s in ["validation", "test"]})
classes.index.name = "class_label"
classes["% validación"] = (classes["validation"] / classes["validation"].sum() * 100).round(2)
classes["% test"] = (classes["test"] / classes["test"].sum() * 100).round(2)
classes

### Coste de referencia: predecir siempre la clase 0

Es el coste de no avisar nunca (hipótesis H5 del E3). Cualquier modelo útil tiene que quedar por debajo.

In [ ]:
def total_cost(y_true, y_pred, matrix=data.COST_MATRIX):
    m = np.asarray(matrix)
    return int(m[np.asarray(y_true), np.asarray(y_pred)].sum())

for s in ["validation", "test"]:
    y = splits[s][2]["class_label"]
    print(f"{s:<10} coste Scania: {total_cost(y, np.zeros_like(y)):>7,}   errores (matriz de 1s): {(y != 0).sum()}")

## 9. Lecturas por camión

In [ ]:
per_vehicle = pd.DataFrame({s: r.groupby("vehicle_id").size().describe() for s, (r, _, _) in splits.items()}).T
per_vehicle

In [ ]:
r = splits["train"][0]
n_read = r.groupby("vehicle_id").size()
for k in [3, 5, 10, 20]:
    print(f"Camiones de train con menos de {k:>2} lecturas: {(n_read < k).sum():>5} ({(n_read < k).mean():.1%})")

## 10. Resumen

Resultados de las comprobaciones (se rellenan al ejecutar):

In [ ]:
r_tr = splits["train"][0]
summary = {
    "CSV completos": bool(raw_status["ok"].all()),
    "ids coherentes en los 3 ficheros": bool(id_check["mismos ids en los 3"].all()),
    "sin camiones compartidos entre conjuntos": not (ids["train"] & ids["validation"] or ids["train"] & ids["test"] or ids["validation"] & ids["test"]),
    "filas o (vehicle_id, time_step) duplicados": int(order_check[["filas duplicadas", "(vehicle_id, time_step) repetidos"]].values.sum()),
    "descensos de contadores en train": int(decreases["descensos"]["train"].sum()),
    "máx % nulos en una columna": round(float(null_pct.values.max()), 2),
    "camiones de train con reparación (%)": round(tte["in_study_repair"].mean() * 100, 2),
}
pd.Series(summary, name="valor").to_frame()